In [ ]:
!mkdir -p ~/.kaggle
!echo KGAT_26d0b434eeea07da0f48bb9e6cee272e > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
!pip install -q kaggle

In [ ]:
!kaggle datasets list -s "lending club"

ref                                                             title                                                  size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------  -----------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
wordsforthewise/lending-club                                    All Lending Club loan data                       1356507910  2019-04-10 18:03:34.347000         112926        924  0.75             
adarshsng/lending-club-loan-data-csv                            Lending Club Loan Data                            355544101  2021-06-17 11:06:05.723000          24888        156  0.9411765        
ethon0426/lending-club-20072020q1                               Lending Club 2007-2020Q3                          505408012  2020-12-15 18:15:28.060000          20475        127  0.9411765        
urstrulyvikas/l

In [ ]:
!kaggle datasets download -d imsparsh/lending-club-loan-dataset-2007-2011
!unzip -q lending-club-loan-dataset-2007-2011.zip -d lending_club_real


Dataset URL: https://www.kaggle.com/datasets/imsparsh/lending-club-loan-dataset-2007-2011
License(s): CC0-1.0
100% 8.39M/8.39M [00:00<00:00, 72.9MB/s]



In [ ]:
import os
os.listdir('lending_club_real')

['loan.csv', 'Data_Dictionary.xlsx']

In [ ]:
import pandas as pd
df = pd.read_csv('lending_club_real/loan.csv', low_memory=False)
df.shape

(39717, 111)

In [ ]:
cols = ['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length',
        'home_ownership', 'annual_inc', 'purpose', 'addr_state', 'dti',
        'delinq_2yrs', 'open_acc', 'issue_d', 'loan_status']
df = df[cols]
df.shape

(39717, 15)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39717 entries, 0 to 39716
Columns: 111 entries, id to total_il_high_credit_limit
dtypes: float64(74), int64(13), object(24)
memory usage: 33.6+ MB


In [ ]:
df['int_rate'].head()
df['term'].unique()
df['home_ownership'].unique()
df['purpose'].unique()

array(['credit_card', 'car', 'small_business', 'other', 'wedding',
       'debt_consolidation', 'home_improvement', 'major_purchase',
       'medical', 'moving', 'vacation', 'house', 'renewable_energy',
       'educational'], dtype=object)

In [ ]:
df['int_rate'].head()


,int_rate
0,10.65%
1,15.27%
2,15.96%
3,13.49%
4,12.69%


In [ ]:
df['int_rate'] = df['int_rate'].str.replace('%', '').astype(float)
df['int_rate'].head()

,int_rate
0,10.65
1,15.27
2,15.96
3,13.49
4,12.69


In [ ]:
df['term'].unique()

array([' 36 months', ' 60 months'], dtype=object)

In [ ]:
df['term'] = df['term'].str.strip()
df['term'].unique()

array(['36 months', '60 months'], dtype=object)

In [ ]:
df['home_ownership'].unique()

array(['RENT', 'OWN', 'MORTGAGE', 'OTHER', 'NONE'], dtype=object)

In [ ]:
df['emp_length'].unique()

array(['10+ years', '< 1 year', '1 year', '3 years', '8 years', '9 years',
       '4 years', '5 years', '6 years', '2 years', '7 years', nan],
      dtype=object)

In [ ]:
df['emp_length'].mode()

,emp_length
0,10+ years


In [ ]:
df['emp_length'].value_counts()

,count
emp_length,
10+ years,8879
< 1 year,4583
2 years,4388
3 years,4095
4 years,3436
5 years,3282
1 year,3240
6 years,2229
7 years,1773


In [ ]:
df['emp_length'] = df['emp_length'].fillna('Unknown')
df['emp_length'].value_counts()

,count
emp_length,
10+ years,8879
< 1 year,4583
2 years,4388
3 years,4095
4 years,3436
5 years,3282
1 year,3240
6 years,2229
7 years,1773


In [ ]:
df['loan_status'].value_counts()

,count
loan_status,
Fully Paid,32950
Charged Off,5627
Current,1140


In [ ]:
df['is_default'] = df['loan_status'] == 'Charged Off'
df['is_default'].mean()

np.float64(0.14167736737417227)

...some analysts argue that its good to exclude loans with unfinished outcome, in this case " current"

In [ ]:
finished = df[df['loan_status'] != 'Current']
finished['is_default'].mean()

np.float64(0.14586411592399617)

moves from 0.1416 to 0.145 which is a small difference. therefore I can used full dataset because the rate isnt materially affected.

In [ ]:
default_by_grade = df.groupby('grade')['is_default'].mean()
default_by_grade

,is_default
grade,
A,0.059693
B,0.118552
C,0.166337
D,0.210665
E,0.251583
F,0.304099
G,0.319620


In [ ]:
df.groupby('grade')['is_default'].count()

,is_default
grade,
A,10085
B,12020
C,8098
D,5307
E,2842
F,1049
G,316


In [ ]:
df.groupby('is_default')['annual_inc'].mean()

,annual_inc
is_default,
False,70048.707623
True,62427.298034


In [ ]:
df.groupby('purpose')['is_default'].mean()

,is_default
purpose,
car,0.103292
credit_card,0.105653
debt_consolidation,0.148436
educational,0.172308
home_improvement,0.116599
house,0.154856
major_purchase,0.101509
medical,0.152958
moving,0.157804


Small business stands out as genuinely riskier with the 0.259 which is significantly higher than the rest. converted to percentage that is 26% defaulted.

In [ ]:
df.groupby('purpose')['is_default'].count()

,is_default
purpose,
car,1549
credit_card,5130
debt_consolidation,18641
educational,325
home_improvement,2976
house,381
major_purchase,2187
medical,693
moving,583


In [ ]:
df['issue_d'].head()

,issue_d
0,Dec-11
1,Dec-11
2,Dec-11
3,Dec-11
4,Dec-11


In [ ]:
df['issue_d_dt'] = pd.to_datetime(df['issue_d'], format='%b-%y')
df['issue_d_dt'].head()

,issue_d_dt
0,2011-12-01
1,2011-12-01
2,2011-12-01
3,2011-12-01
4,2011-12-01


In [ ]:
monthly = df.groupby(df['issue_d_dt'].dt.to_period('M')).agg(
    volume=('loan_amnt', 'sum'),
    count=('loan_amnt', 'count')
)
monthly

,volume,count
issue_d_dt,,
2007-06,7500,1
2007-07,171700,30
2007-08,208475,33
2007-09,146025,18
2007-10,318775,47
2007-11,380625,37
2007-12,986175,85
2008-01,1761050,171
2008-02,1679375,174


There is a growth from June 2007 which had one loan of $ 7500 to december 2011 which has 2260 loans amounting to 31.5 million dollars.
There is a significant drop in number of loans borrowed in september 2008 as compared to the beginning of that year. This happened during the 2008 global financial crisis.
"Global credit froze as banks lost trust in each other's solvency, stopping regular lending to businesses and consumers"
Because banks were hoarding cash, they drastically tightened their lending rules. Getting approved for a loan became nearly impossible overnight.